# Customer Churn, baseline local

Este notebook es el puente corto entre Canvas y sistema. Tomamos el dataset canónico, entrenamos un baseline y generamos un ranking por prioridad operativa.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path('../data/raw/Telco-Customer-Churn.csv')
df = pd.read_csv(DATA)
df.shape, df.columns.tolist()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0.0)
df['Churn'].value_counts(normalize=True)

In [ ]:
import sys
sys.path.append('..')

from scripts.train_baseline import build_pipeline, load_training_data
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, recall_score, precision_score

X, y = load_training_data(DATA)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)
proba = pipeline.predict_proba(X_test)[:, 1]
pred = proba >= 0.7
{
    'roc_auc': roc_auc_score(y_test, proba),
    'precision_at_0_7': precision_score(y_test, pred),
    'recall_at_0_7': recall_score(y_test, pred),
}

In [ ]:
scored = X_test.copy()
scored['customerID'] = df.loc[X_test.index, 'customerID']
scored['churn_probability'] = proba
scored['priority_score'] = scored['churn_probability'] * scored['MonthlyCharges']
scored[['customerID', 'churn_probability', 'MonthlyCharges', 'priority_score']].sort_values(
    'priority_score', ascending=False
).head(10)